In [ ]:
from lightkurve import DataCube, ErrorCube
import numpy as np

%load_ext autoreload
%autoreload 2

# Obtain Data

In [ ]:
from lksearch import TESSSearch
from astropy.coordinates import SkyCoord
from astropy.io import fits

search_input = (84.291190, -80.469170)
search_input = SkyCoord(*search_input, unit="deg", frame="icrs")
search_result = TESSSearch(search_input, hlsp=False)

# You can filter two different ways to get the same result.
# search_result.filter_table(pipeline='TESScut')
downloads_table = search_result.tesscut.filter_table(sector=1).download(TESScut_size=10)
hdulist = fits.open(downloads_table["Local Path"][0])

# Create a DataCube

<span style="font-size:1.2em;">
The only mandatory input to create a Cube object is an array of fluxes in the form (n_time, n_row, n_columns).
</span>

In [ ]:
flux_array = hdulist[1].data["FLUX"].astype(float)
flux_array.shape

In [ ]:
flux = DataCube(flux_array)

print(flux)

<span style="font-size:1.2em;">
The standard representation of these objects gives the object type and the shape. In a Jupyter notebook, the output will display a `pandas.DataFrame` for first cadence. 
</span>

In [ ]:
flux

## ErrorCube

<span style="font-size:1.2em;">
ErrorCubes are a subclass of DataCubes. In form and function they are identical to DataCubes, but treat error appropriately when performing statistical operations.
</span>

In [ ]:
flux_err_array = hdulist[1].data["FLUX_ERR"].astype(float)
flux_err = ErrorCube(flux_err_array)
flux_err

#### Meta Information

<span style="font-size:1.2em;">
The <tt>meta</tt> attribute will display some key information including the time indices (in this first example, just the cadences) and the unique row and column IDs.
</span>

In [ ]:
flux.meta

# A more realistic example
<span style="font-size:1.2em;">
Naming the row and columns and accounting for their real pixel positions, multi-indexing with cadence, collection time, and spacecraft time
</span>

In [ ]:
# Reference for the 0-index row and column positions on the CCD
c0, r0 = hdulist[1].header["1CRV4P"], hdulist[1].header["2CRV4P"]
row, col = np.arange(flux_array.shape[1]) + r0, np.arange(flux_array.shape[2]) + c0

time = hdulist[1].data["TIME"].astype(float)
# Spacecraft time correction for relativistic effects
time_corr = hdulist[1].data["TIMECORR"].astype(float)

In [ ]:
flux = DataCube(
    flux_array,
    time_indices={"btjd": time, "spacecraft_time": time - time_corr},
    row_indices={"pixel_row": row},  # optional
    col_indices={"pixel_column": col},  # optional
)

print(flux)

<span style="font-size:1.2em;">
Note that the shapes are the same, but in the Jupyter output, the column and row are renamed and the DataFrame title contains information for all time indices.
</span>

In [ ]:
flux

<span style="font-size:1.2em;">
The header shows the times available and shows the renamed row and columns
</span>

In [ ]:
flux.meta

<span style="font-size:1.2em;">
Note that these attributes are accessible directly from the DataCube object by the names <i>you provide</i> (not accessible through the header)
</span>

In [ ]:
flux.cadence

In [ ]:
flux.btjd

In [ ]:
flux.spacecraft_time

<span style="font-size:1.2em;">
Note that the rows and columns are, in practice, flattened. As such, the <tt>pixel_row</tt> and <tt>pixel_column</tt> attributes, if accessed directly, may not match what you'd expect to see.
</span>

In [ ]:
flux.shape

In [ ]:
flux.pixel_row

In [ ]:
flux.pixel_column

Accessing flux for a particular pixel or set of pixels expects slices or indices for cadence, row, and column.

In [ ]:
flux[0, 0, 0]

In [ ]:
flux[:10, 0, 0]

In [ ]:
flux[:10, 0, :5]

# Next Up:

* [Apertures, masks, and downsampling](apertures-masks-downsampling.ipynb)
* [Core Functionality](datacube-operations.ipynb)